# 2 — A freshly generated dataset with no annotation

If nothing is annotated yet there is nothing to audit, so start with proposals.

> **This is a curation aid, not an annotator.** Measured against 200 hand-curated cell
> types with no label involved, the top-scoring term is right **55%** of the time
> (35% bone marrow, 86% pancreas), and a right call cannot be told from a wrong one
> (leave-one-organ-out AUC **0.563**). There is no threshold that makes it safe to
> automate. For automated annotation use a trained classifier — CellTypist, Azimuth,
> popV — and then audit the result.

What it *is* good for: the correct term is in the **top five 73–100%** of the time, so
it turns "name this cluster" into "choose among five, with evidence".

## celltype-audit does not cluster

Cluster first, with scanpy or Seurat. If no cluster column is found, celltype-audit says
so rather than guessing.

In [ ]:
# import scanpy as sc
# adata = sc.read_h5ad('fresh.h5ad')
# sc.pp.neighbors(adata); sc.tl.leiden(adata)
# adata.write('fresh_clustered.h5ad')

In [ ]:
from celltype_audit import annotate_h5ad

doc = annotate_h5ad(
    'fresh_clustered.h5ad',
    cluster_key='leiden',
    tissue='UBERON:0002048',   # needed if the file carries no tissue id
)
print(doc['meta']['accuracy'])

In [ ]:
for c in doc['clusters'][:8]:
    top = c['proposals'][0]['label'] if c['proposals'] else '(%s)' % c['status']
    print('cluster %-6s %7d  %s' % (c['cluster'], c['n_cells'], top))
    for p in c['proposals'][1:4]:
        print('%22s %d. %-42s %.2f' % ('', p['rank'], p['label'], p['relative']))
    print('%22s markers: %s' % ('', ', '.join(c['markers'][:6])))

## Three fields to read

- **`relative`** — each proposal against the top one. Values near 1.0 mean the
  shortlist does not discriminate and the cluster needs a human.
- **`lineage_agreement`** — whether the whole top-N sits in one lineage. `False` means
  the evidence has not even settled the broad class.
- **`status`** — why a cluster got nothing, when it does. There is never a silent blank.

`tissue_declared` / `rolled_up` record when a cluster's own sub-tissue (say *cortex of
kidney*) had no reference and was rolled up to the part that does (*kidney*). That is a
`part_of` relation, reported rather than applied silently.

In [ ]:
amb = [c for c in doc['clusters']
       if c['proposals'] and c['proposals'][-1]['relative'] > 0.85]
print('%d of %d clusters have an undiscriminating shortlist' % (amb.__len__(), len(doc['clusters'])))
for c in amb[:5]:
    print(' ', c['cluster'], [p['label'] for p in c['proposals'][:3]])

## The honest workflow

```
cluster  ->  annotate with a trained classifier  ->  celltype-audit audit
```

Use `annotate` to speed manual curation, not to replace it. Step 3 is where this
package earns its keep.